In [20]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

In [21]:
DATA_PATH  = "data/data_sample.csv"
PROCESSED_DIR  = "data/processed"
MODELS_DIR     = "models"

In [22]:
TIMESTAMP_COL  = "datetime_from"
# FEATURE_COLS   = ["load","res","ppt","borders","curtailments","holiday"]
FEATURE_COLS   = ["load","res","ppt","borders","curtailments","holiday","day_of_week_Friday","day_of_week_Monday","day_of_week_Saturday","day_of_week_Sunday","day_of_week_Thursday","day_of_week_Tuesday","day_of_week_Wednesday"]
TARGET_COL     = "curtailments"    
SEQ_LEN        = 96    
TRAIN_RATIO    = 0.70
VAL_RATIO      = 0.15 

In [23]:
def load_and_validate(path: str) -> pd.DataFrame:
    print(f"[1/5] Loading data from: {path}")
    df = pd.read_csv(path, parse_dates=[TIMESTAMP_COL])
    df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    # Basic checks
    assert len(df) >= SEQ_LEN * 10, \
        f"Too few rows ({len(df)}). Need at least {SEQ_LEN * 10}."
    missing = df[FEATURE_COLS + [TARGET_COL]].isnull().sum()
    if missing.any():
        print(f"  WARNING: Missing values detected:\n{missing[missing > 0]}")
        print("  Filling with forward fill + backward fill.")
        df[FEATURE_COLS + [TARGET_COL]] = (
            df[FEATURE_COLS + [TARGET_COL]]
            .ffill()
            .bfill()
        )

    n_full_days = len(df) // SEQ_LEN
    n_dropped   = len(df) - n_full_days * SEQ_LEN
    print(f"  Rows: {len(df)} | Full days: {n_full_days} | "
          f"Dropped (partial day): {n_dropped} rows")
    return df.iloc[: n_full_days * SEQ_LEN].reset_index(drop=True)

In [24]:
def reshape_to_days(df: pd.DataFrame):
    print(f"[2/5] Reshaping to (n_days, {SEQ_LEN}, n_features) ...")
    n_days = len(df) // SEQ_LEN
    n_feat = len(FEATURE_COLS)

    X = df[FEATURE_COLS].values.reshape(n_days, SEQ_LEN, n_feat)  # (n_days, 96, 7)
    y = df[TARGET_COL].values.reshape(n_days, SEQ_LEN)             # (n_days, 96)

    print(f"  X shape: {X.shape}  |  y shape: {y.shape}")
    return X, y

In [25]:
def split_chronological(X: np.ndarray, y: np.ndarray):
    print("[3/5] Splitting train / val / test (no shuffle) ...")
    n        = len(X)
    n_train  = int(n * TRAIN_RATIO)
    n_val    = int(n * (TRAIN_RATIO + VAL_RATIO))

    splits = {
        "train": (X[:n_train],       y[:n_train]),
        "val":   (X[n_train:n_val],  y[n_train:n_val]),
        "test":  (X[n_val:],         y[n_val:]),
    }

    for name, (Xs, ys) in splits.items():
        print(f"  {name:5s}: {len(Xs):4d} days  "
              f"({len(Xs)/n*100:.1f}%)")
    return splits

In [26]:
def scale(splits: dict):
    """
    Fit StandardScaler on TRAIN only. Apply to all splits.
    - scaler_X: fitted on (n_train * 96, 7) — scales each feature independently
    - scaler_y: fitted on (n_train, 96)     — scales each timestep independently
    Test y is saved in BOTH scaled (for loss checks) and
    raw (for final metric evaluation) form.
    """
    print("[4/5] Scaling features and target ...")
    X_train, y_train = splits["train"]
    n_feat = X_train.shape[2]

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    # Fit only on training data
    scaler_X.fit(X_train.reshape(-1, n_feat))
    scaler_y.fit(y_train)

    scaled = {}
    for name, (X, y) in splits.items():
        X_sc = scaler_X.transform(
            X.reshape(-1, n_feat)
        ).reshape(X.shape).astype("float32")

        y_sc = scaler_y.transform(y).astype("float32")

        scaled[name] = (X_sc, y_sc)
        print(f"  {name:5s}: X {X_sc.shape}  y {y_sc.shape}")

    # Keep raw (unscaled) test target for metric evaluation
    scaled["test_y_raw"] = splits["test"][1].astype("float32")

    return scaled, scaler_X, scaler_y

In [27]:
def save(scaled: dict, scaler_X, scaler_y):
    print("[5/5] Saving to disk ...")
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    os.makedirs(MODELS_DIR, exist_ok=True)

    for name, data in scaled.items():
        if name == "test_y_raw":
            path = os.path.join(PROCESSED_DIR, "test_y_raw.npy")
            np.save(path, data)
            print(f"  Saved: {path}")
        else:
            X, y = data
            for arr, tag in [(X, "X"), (y, "y")]:
                path = os.path.join(PROCESSED_DIR, f"{tag}_{name}.npy")
                np.save(path, arr)
                print(f"  Saved: {path}")

    # Scalers shared by all three models at inference time
    joblib.dump(scaler_X, os.path.join(MODELS_DIR, "scaler_X.pkl"))
    joblib.dump(scaler_y, os.path.join(MODELS_DIR, "scaler_y.pkl"))
    print(f"  Saved: {MODELS_DIR}/scaler_X.pkl")
    print(f"  Saved: {MODELS_DIR}/scaler_y.pkl")


# ── SUMMARY ───────────────────────────────────────────────────────────────────
def print_summary():
    print("\n" + "="*50)
    print("  PREPROCESSING COMPLETE")
    print("="*50)
    print("  Files produced:")
    print(f"    {PROCESSED_DIR}/X_train.npy,  y_train.npy")
    print(f"    {PROCESSED_DIR}/X_val.npy,    y_val.npy")
    print(f"    {PROCESSED_DIR}/X_test.npy,   y_test.npy")
    print(f"    {PROCESSED_DIR}/test_y_raw.npy  (unscaled test target)")
    print(f"    {MODELS_DIR}/scaler_X.pkl,  scaler_y.pkl")
    print("\n  These files are shared by Ridge, XGBoost, and Bi-LSTM.")
    print("  Next step:  python src/train.py --model ridge")
    print("="*50)


In [29]:
if __name__ == "__main__":
    df              = load_and_validate(os.path.join(os.getcwd(), "..",DATA_PATH))
    X, y            = reshape_to_days(df)
    splits          = split_chronological(X, y)
    scaled, sx, sy  = scale(splits)
    save(scaled, sx, sy)
    print_summary()

[1/5] Loading data from: /Users/vassilikimpelogianni/Documents/Code/res_curtailements_prediction/src/../data/data_sample.csv
  Rows: 49532 | Full days: 515 | Dropped (partial day): 92 rows
[2/5] Reshaping to (n_days, 96, n_features) ...
  X shape: (515, 96, 13)  |  y shape: (515, 96)
[3/5] Splitting train / val / test (no shuffle) ...
  train:  360 days  (69.9%)
  val  :   77 days  (15.0%)
  test :   78 days  (15.1%)
[4/5] Scaling features and target ...
  train: X (360, 96, 13)  y (360, 96)
  val  : X (77, 96, 13)  y (77, 96)
  test : X (78, 96, 13)  y (78, 96)
[5/5] Saving to disk ...
  Saved: data/processed/X_train.npy
  Saved: data/processed/y_train.npy
  Saved: data/processed/X_val.npy
  Saved: data/processed/y_val.npy
  Saved: data/processed/X_test.npy
  Saved: data/processed/y_test.npy
  Saved: data/processed/test_y_raw.npy
  Saved: models/scaler_X.pkl
  Saved: models/scaler_y.pkl

  PREPROCESSING COMPLETE
  Files produced:
    data/processed/X_train.npy,  y_train.npy
    data/p